In [ ]:
# Checking .yaml load/

In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

In [3]:
initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [4]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


# TODO: make it so that everything uses the config file:
redo the 'return path' functions, and rely on config file if not given input :)and rely of n config file if not given input :).
prompted gpt 'april 8' so can check that chat at the bottom I think its pretty good 

In [5]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [6]:
# TODO: make functions less verbose for data creation
# TODO: clean import statments
# TODO: see if we are calculating Total Energy correctly, I think the whole row is all 0 after std so ... 
# TODO: Experiment with the derivatives and  not derivatives data (preprocessed and 'raw' respectively, I think currently raw_

In [7]:
 # this is a good little tutorial to understand basics of pyspark  
# https://domino.ai/blog/principal-component-analysis-pca-on-large-neuroimaging-datasets-using-pyspark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [9]:
# Spark is a library that distributes the load of computation/ram very efficiently and evenly :)

In [10]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [11]:
# # Set environment variables
# os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# # Create new session with explicit local binding
# spark = SparkSession.builder \
#     .appName("EEG_Analysis") \
#     .config("spark.driver.bindAddress", "127.0.0.1") \
#     .config("spark.driver.host", "127.0.0.1") \
#     .master("local[*]") \
#     .getOrCreate()


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

# print("New Spark session created successfully")

In [12]:
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \
    
    # to not reach 100% CPU utilizatoin and get stuck
    .config("spark.master", "local[8]")

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)
# Enable Apache Arrow for pandas UDF performance boost
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/24 01:27:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


In [13]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction_helper.py"))
    print("Added feature_extraction_helper.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added feature_extraction_helper.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [14]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participantsfname
subject_df = subject_df.repartition(200, "SubjectID")  # tweak 200 based on cluster resources

In [15]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-003"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
print("3 tables types")
sub1.filter((sub1.table_type=="band")).show(3)
sub1.filter((sub1.table_type=="electrode")).show(3)
sub1.filter((sub1.table_type=="epoch")).show(3)

/Users/admin/eeg-ds004504
3 tables types


/Volumes/CrucialX6/Home/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py=======================>(99 + 1) / 100]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
[START] sub-003
processSub sub-003
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-003
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000g

+---------+-------+---------+--------+---------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|    FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+---------------+------------+----------+
|  sub-003|   ep-0|      Fp1|   Alpha|          Power| 0.008017266|      band|
|  sub-003|   ep-0|      Fp1|   Alpha|SpectralEntropy|   2.2665515|      band|
|  sub-003|   ep-0|      Fp1|   Alpha| HjorthMobility| 0.045576047|      band|
+---------+-------+---------+--------+---------------+------------+----------+
only showing top 3 rows



Config not found in feature_extraction.py=======================>(99 + 1) / 100]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
[START] sub-003
processSub sub-003
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-003
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-ee5df60b0036/preprocess_sets.py:108: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw =

+---------+-------+---------+--------+---------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|    FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+---------------+------------+----------+
|  sub-003|   ep-0|      Fp1|    NULL|     TotalPower| 0.011235955| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|    TotalEnergy|  0.10781828| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|SpectralEntropy|   4.1740084| electrode|
+---------+-------+---------+--------+---------------+------------+----------+
only showing top 3 rows



Config not found in feature_extraction.py=======================>(99 + 1) / 100]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
[START] sub-003
processSub sub-003
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-003
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-ee5df60b0036/preprocess_sets.py:108: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw =

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-003|   ep-0|     NULL|    NULL|       Mean|7.329646E-21|     epoch|
|  sub-003|   ep-0|     NULL|    NULL|        Std| 3.055757E-5|     epoch|
|  sub-003|   ep-0|     NULL|    NULL|   Variance|9.346697E-10|     epoch|
+---------+-------+---------+--------+-----------+------------+----------+
only showing top 3 rows

CPU times: user 31.3 ms, sys: 26 ms, total: 57.3 ms
Wall time: 2min 7s


In [16]:
sub1.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- EpochID: string (nullable = false)
 |-- Electrode: string (nullable = true)
 |-- WaveBand: string (nullable = true)
 |-- FeatureName: string (nullable = true)
 |-- FeatureValue: float (nullable = true)
 |-- table_type: string (nullable = true)



In [17]:
sub1.show()

Config not found in feature_extraction.py=======================>(99 + 1) / 100]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
[START] sub-003
processSub sub-003
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-003
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-ee5df60b0036/preprocess_sets.py:108: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw =

+---------+-------+---------+--------+----------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|     FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+----------------+------------+----------+
|  sub-003|   ep-0|      Fp1|    NULL|      TotalPower| 0.011235955| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|     TotalEnergy|  0.10781828| electrode|
|  sub-003|   ep-0|      Fp1|    NULL| SpectralEntropy|   4.1740084| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|  HjorthActivity|9.705875E-10| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|  HjorthMobility| 0.045576047| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|HjorthComplexity|   5.9060574| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|     HjorthIndex|   1108.8796| electrode|
|  sub-003|   ep-0|      Fp1|   Alpha|           Power| 0.008017266|      band|
|  sub-003|   ep-0|      Fp1|   Alpha| SpectralEntropy|   2.2665515|      band|
|  sub-003|   ep-0|      Fp1|   Alpha|  

In [19]:
# print(f"There are {subject_df.filter(subject_df.Group == "C").count()} control

In [22]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
from functools import reduce
from pyspark.sql import DataFrame

group_a_subjects = (
    subject_df.filter(subject_df.Group == "A")
    .select("SubjectID")
    .distinct()
    .rdd.flatMap(lambda r: r)
    .collect()
)

group_c_subjects = (
    subject_df.filter(subject_df.Group == "C")
    .select("SubjectID")
    .distinct()
    .rdd.flatMap(lambda r: r)
    .collect()
)

results_a = []
for subj in group_a_subjects:
    print(f"\n=== Processing A: {subj} ===")
    try:
        df = (
            subject_df
            .filter(subject_df.SubjectID == subj)
            .groupBy("SubjectID")
            .apply(extract_features_udtf)
        ).persist()
        
        _ = df.count()  # ✅ Force execution right now
        results_a.append(df)
        print(f"✅ Finished A: {subj}")
    except Exception as e:
        print(f"[ERROR] Subject A {subj} failed: {e}")

results_c = []
for subj in group_c_subjects:
    print(f"\n=== Processing C: {subj} ===")
    try:
        df = (
            subject_df
            .filter(subject_df.SubjectID == subj)
            .groupBy("SubjectID")
            .apply(extract_features_udtf)
        ).persist()
        
        _ = df.count()  # ✅ Force execution right now
        results_c.append(df)
        print(f"✅ Finished C: {subj}")
    except Exception as e:
        print(f"[ERROR] Subject C {subj} failed: {e}")

result_group_a = reduce(DataFrame.unionByName, results_a)
result_group_c = reduce(DataFrame.unionByName, results_c)

print(f"\n✅ All Group A subjects done ({len(results_a)} total)")
print(f"✅ All Group C subjects done ({len(results_c)} total)")


# group_a_spark_df = (
#     subject_df
#     .filter(subject_df.Group == "A")
#     .groupBy("SubjectID")
#     .apply(extract_features_udtf)
# )

# group_c_spark_df = (
#     subject_df
#     .filter(subject_df.Group == "C")
#     .groupBy("SubjectID")
#     .apply(extract_features_udtf)
# )


# # Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
# result_group_a = group_a_spark_df.persist()
# result_group_c = group_c_spark_df.persist()

# # Trigger execution with actions
# count_a = result_group_a.count()
# count_c = result_group_c.count()

# print(f"Processed {count_a} records for Alzheimer's group")
# print(f"Processed {count_c} records for Control group")





=== Processing A: sub-020 ===


Config not found in feature_extraction.py======================>(199 + 1) / 200]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
[START] sub-020
processSub sub-020
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-020
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-020/eeg/sub-020_task-eyesclosed_eeg.set
[FINISHED] sub-020 in 251.98s==================================>(199 + 1) / 200]
[START] sub-033                                                                 
processSub sub-033
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-033
subPath data_path /Users/admin/eeg-ds0045

✅ Finished A: sub-020

=== Processing A: sub-033 ===


/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-ee5df60b0036/preprocess_sets.py:108: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(subPath(sub, derivatives), preload=True)
[FINISHED] sub-033 in 169.40s==================================>(199 + 1) / 200]
[START] sub-012                                                                 
processSub sub-012
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-012
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-012/eeg/sub-012_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-

✅ Finished A: sub-033

=== Processing A: sub-012 ===


...

✅ Finished C: sub-038

=== Processing C: sub-044 ===


[FINISHED] sub-044 in 254.39s==================================>(199 + 1) / 200]
[START] sub-040                                                                 
processSub sub-040
processSub: derivatives True
processSub: windowLength 3
processSub: windowLength 0.3
subPath sub-040
subPath data_path /Users/admin/eeg-ds004504
Derivatives: True
Derivatives from config: True
Path handed: /Users/admin/eeg-ds004504/ds004504/derivatives/sub-040/eeg/sub-040_task-eyesclosed_eeg.set
/private/var/folders/l1/yg4p3qp10vq4pn5lyx21tcdr0000gq/T/spark-12690d58-93e2-4915-b9d6-50a5cc609eda/userFiles-f14233a6-f94e-4add-ae03-ee5df60b0036/preprocess_sets.py:108: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(subPath(sub, derivatives), preload=True)


✅ Finished C: sub-044

=== Processing C: sub-040 ===


[FINISHED] sub-040 in 233.31s==================================>(199 + 1) / 200]
                                                                                

✅ Finished C: sub-040

✅ All Group A subjects done (36 total)
✅ All Group C subjects done (29 total)
CPU times: user 1.39 s, sys: 1.1 s, total: 2.49 s
Wall time: 3h 56min 54s


In [44]:
 alz_df_spark = result_group_a 
 cntrl_df_spark = result_group_c

In [45]:
type(alz_df_spark)

pyspark.sql.dataframe.DataFrame

In [28]:
print("worked")

worked


In [29]:
result_group_a.columns

['SubjectID',
 'EpochID',
 'Electrode',
 'WaveBand',
 'FeatureName',
 'FeatureValue',
 'table_type']

In [ ]:
print(result_group_a.select("table_type", "FeatureName").groupBy("table_type").count().show())
print(result_group_c.select("table_type", "FeatureName").groupBy("table_type").count().show())


In [ ]:
print(result_group_a.select("table_type", "FeatureName").groupBy("table_type").count().show())

In [ ]:
#Since we doni't want to recreate the data all the time, lets save it and I will see you in Example_Data_Processing

In [31]:
type(result_group_a)

pyspark.sql.dataframe.DataFrame

In [46]:
group_a_pandas_df = alz_df_spark.toPandas() 
group_c_pandas_df = cntrl_df_spark.toPandas() 


25/04/24 08:21:27 ERROR TaskSetManager: Total size of serialized results of 2250 tasks (1049.7 MiB) is bigger than spark.driver.maxResultSize (1024.0 MiB)
25/04/24 08:21:27 ERROR TaskSetManager: Total size of serialized results of 2251 tasks (1049.7 MiB) is bigger than spark.driver.maxResultSize (1024.0 MiB)
25/04/24 08:21:27 WARN TaskSetManager: Lost task 2249.0 in stage 875.0 (TID 60627) (a-10-27-14-45.dynapool.vpn.nyu.edu executor driver): TaskKilled (Tasks result size has exceeded maxResultSize)
25/04/24 08:21:27 WARN TaskSetManager: Lost task 1859.0 in stage 875.0 (TID 60237) (a-10-27-14-45.dynapool.vpn.nyu.edu executor driver): TaskKilled (Tasks result size has exceeded maxResultSize)
25/04/24 08:21:27 ERROR TaskSetManager: Total size of serialized results of 2252 tasks (1049.7 MiB) is bigger than spark.driver.maxResultSize (1024.0 MiB)
25/04/24 08:21:27 WARN TaskSetManager: Lost task 2248.0 in stage 875.0 (TID 60626) (a-10-27-14-45.dynapool.vpn.nyu.edu executor driver): TaskKill

Py4JJavaError: An error occurred while calling o2065.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j

In [47]:
from datetime import datetime
import os
from config_handler import load_config

def create_database_log(filename, config, timestamp, df=None):
    """
    Creates a log file with information about the saved dataframe.
    
    Parameters:
    - df: The Spark DataFrame being saved
    - filename: The name of the saved pkl file
    - config: Configuration dictionary from load_config()
    - timestamp: Timestamp string for the log entry
    """
    # Count unique subjects
    try:
        unique_subjects = df.select("SubjectID").distinct().count()
    except:
        unique_subjects = -1
    
    # Create log entry
    log_entry = [
        f"=== DATABASE LOG ENTRY: {timestamp} ===",
        f"Saved file: {filename}",
        f"Unique subjects: {unique_subjects}",
        "\nConfiguration:"
    ]
    
    # Add configuration details
    for key, value in config.items():
        if key == "freqBands":
            log_entry.append(f"  {key}:")
            for band, freq_range in value.items():
                log_entry.append(f"    {band}: {freq_range}")
        else:
            log_entry.append(f"  {key}: {value}")
    
    log_entry.append("\n")
    
    # Write to log file
    with open("databaselog.txt", "a") as f:
        f.write("\n".join(log_entry))
    
    print(f"Log entry written to databaselog.txt")

In [48]:
from datetime import datetime
from config_handler import load_config

# Generate a timestamp like "Apr14_2230"
timestamp = datetime.now().strftime("%b%d_%H%M")

# Filenames with timestamp
alz_filename = f"features_alz_extra_features_{timestamp}"
cntrl_filename = f"features_cntrl_extra_features_{timestamp}"

# Save with timestamp in filename
# group_a_pandas_df.to_pickle(f"{alz_filename}.pkl")
# group_c_pandas_df.to_pickle(f"{cntrl_filename}.pkl")

# Get configuration once
config = load_config()

# Create log entries for both files
create_database_log(alz_filename, config, timestamp)
create_database_log(cntrl_filename, config, timestamp)

Log entry written to databaselog.txt
Log entry written to databaselog.txt


In [49]:
# Save alz_df_spark to Parquet
alz_df_spark.write.mode("overwrite").parquet(f"{alz_filename}.parquet")

# Save cntrl_df_spark to Parquet
cntrl_df_spark.write.mode("overwrite").parquet(f"{cntrl_filename}.parquet")

In [ ]:
group_a_pandas_df.to_parquet(f"{alz_filename}_pandas.parquet")
group_c_pandas_df.to_parquet(f"{cntrl_filename}_pandas.parquet")

In [56]:
alz_df_spark.show(1)

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-020|   ep-0|      Fp1|    NULL| TotalPower| 0.011235955| electrode|
+---------+-------+---------+--------+-----------+------------+----------+
only showing top 1 row



# checking the difference between pkl files

In [57]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas


group_a_parquet_df_loaded = spark.read.parquet(f"{alz_filename}.parquet")
group_c_parquet_df_loaded = spark.read.parquet(f"{cntrl_filename}.parquet")


In [58]:
type(group_a_parquet_df_loaded)

pyspark.sql.dataframe.DataFrame

In [54]:
if group_a_parquet_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 1095:================>                                   

Correctly pkl'd and correcfly loaded into pyspark object


In [ ]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)

# checking if the config version same as previous version

In [ ]:
import pandas as pd
post_conf_a = pd.read_pickle("features_alz_example_post_config.pkl")
pre_conf_a = pd.read_pickle("features_alz_example.pkl")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd


In [ ]:
post_conf_a = spark.createDataFrame(post_conf_a)
pre_conf_a = spark.createDataFrame(pre_conf_a)

In [ ]:
if post_conf_a.exceptAll(pre_conf_a).isEmpty():
    print("pre and post are equal")

In [ ]:
spark.stop()